In [ ]:
import glob
import os

import pandas as pd
from PIL import Image

IMAGE_FOLDER = "data/images/train"    # UBAH
LABEL_FOLDER = "data/labels/train"    # UBAH
CLASS_NAMES = ["entete", "vin"]       # UBAH: nomor kelas 0, 1, ...

image_rows = []
box_rows = []
for path in sorted(glob.glob(f"{IMAGE_FOLDER}/*")):
    width, height = Image.open(path).size
    image_rows.append([path, width, height])
    label_file = f"{LABEL_FOLDER}/{os.path.splitext(os.path.basename(path))[0]}.txt"
    if not os.path.exists(label_file):
        continue                      # gambar tanpa kotak
    for line in open(label_file).read().split("\n"):
        if line.strip() == "":
            continue
        number, x_center, y_center, box_width, box_height = line.split()[:5]
        x_center, box_width = float(x_center) * width, float(box_width) * width
        y_center, box_height = float(y_center) * height, float(box_height) * height
        box_rows.append([path, CLASS_NAMES[int(number)], x_center - box_width / 2,
                         y_center - box_height / 2, x_center + box_width / 2,
                         y_center + box_height / 2])
images = pd.DataFrame(image_rows, columns=["path", "width", "height"])
boxes = pd.DataFrame(box_rows, columns=["path", "label", "x1", "y1", "x2", "y2"])
print("gambar:", len(images), "| kotak:", len(boxes))


In [ ]:
print("kotak per kelas:")
print(boxes["label"].value_counts())
per_image = boxes.groupby(["path", "label"]).size()
print("kotak per gambar per kelas (berapa gambar):")
print(per_image.value_counts())
no_box = images[~images["path"].isin(boxes["path"])]
print("gambar tanpa kotak:", len(no_box))
image_area = dict(zip(images["path"], images["width"] * images["height"]))
box_area = (boxes["x2"] - boxes["x1"]) * (boxes["y2"] - boxes["y1"])
boxes["relative_area"] = box_area / boxes["path"].map(image_area)   # 0.01 = 1% luas gambar
area_summary = boxes.groupby("label")["relative_area"].describe()
print(area_summary.round(3))


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

N_SHOW = 8   # UBAH
colors = ["red", "lime", "blue", "yellow", "cyan", "magenta"]
class_list = sorted(boxes["label"].unique())
fig, axes = plt.subplots(2, N_SHOW // 2, figsize=(4 * N_SHOW // 2, 8))
for ax, path in zip(axes.ravel(), images["path"].sample(N_SHOW, random_state=0)):
    ax.imshow(Image.open(path).convert("RGB"))
    for box in boxes[boxes["path"] == path].itertuples():
        color = colors[class_list.index(box.label) % len(colors)]
        ax.add_patch(plt.Rectangle((box.x1, box.y1), box.x2 - box.x1, box.y2 - box.y1,
                                   fill=False, color=color, linewidth=2))
        ax.text(box.x1, box.y1, box.label, color=color, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()
